# FBF-IIoT Complete Reproducible Implementation

This notebook runs the GitHub implementation of federated learning, BRAGA Byzantine robust aggregation, adaptive DP/CKKS privacy, hybrid PBFT/PoA consensus, audit logging, robustness experiments, confidence intervals, and consensus benchmarking.

**Integrity rule:** source benchmark datasets are required for research runs. Missing datasets are not replaced with synthetic paper data, and system metrics are measured by the implementation rather than hard coded.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from fbf_iiot.config import ExperimentConfig, PrivacyConfig, AggregationConfig, NetworkConfig
from fbf_iiot.data import load_dataset
from fbf_iiot.experiment import run_experiment
from fbf_iiot.plotting import plot_convergence, plot_robustness
from fbf_iiot.statistics import summarize_final_rounds

print("Repository root:", ROOT.resolve())

## Dataset audit

The loaders use source data from SECOM, NASA C-MAPSS FD001, Tennessee Eastman Process, and FactoryNet; they do not create local synthetic substitutes under those dataset names. The quick audit below loads SECOM and prints its actual shape and class balance.

In [ ]:
bundle = load_dataset("secom", cache_dir=ROOT / "data" / "cache", max_samples=10000, seed=42)
print("Dataset:", bundle.name)
print("Source:", bundle.source)
print("Shape:", bundle.X.shape)
print("Positive class ratio:", float(bundle.y.mean()))

## End to end FBF-IIoT run

This configuration uses non IID clients, BRAGA aggregation, hybrid consensus, and no privacy transformation so that the full execution path can be verified without optional HE dependencies.

In [ ]:
quick_exp = ExperimentConfig(
    dataset="secom",
    max_samples=10000,
    clients=5,
    rounds=5,
    local_epochs=1,
    batch_size=64,
    dirichlet_alpha=0.5,
    min_client_samples=20,
    aggregator="braga",
    consensus="hybrid",
    validators=4,
    attack="sign_flip",
    attack_fraction=0.20,
    attack_strength=5.0,
    seed=42,
    device="auto",
)
quick_privacy = PrivacyConfig(mode="none")
quick_aggregation = AggregationConfig(method="braga")
quick_network = NetworkConfig(base_latency_ms=2.0, jitter_ms=0.5, packet_loss=0.0)

history, client_history, run_info = run_experiment(
    quick_exp,
    quick_privacy,
    quick_aggregation,
    quick_network,
    cache_dir=str(ROOT / "data" / "cache"),
    out_dir=ROOT / "results" / "notebook_quick_run",
)

display(history.tail())
print(run_info)

## FedAvg versus BRAGA under Byzantine attacks

This compact experiment measures the contribution of BRAGA at increasing malicious client ratios. For final paper tables use the publication script with all datasets and five seeds.

In [ ]:
from dataclasses import replace

frames = []
for aggregator in ["fedavg", "braga"]:
    for attack_fraction in [0.0, 0.1, 0.2, 0.3]:
        for seed in [42, 43, 44]:
            exp = replace(
                quick_exp,
                rounds=10,
                aggregator=aggregator,
                attack="none" if attack_fraction == 0 else "sign_flip",
                attack_fraction=attack_fraction,
                seed=seed,
            )
            agg = AggregationConfig(method=aggregator)
            h, _, _ = run_experiment(
                exp,
                PrivacyConfig(mode="none"),
                agg,
                quick_network,
                cache_dir=str(ROOT / "data" / "cache"),
            )
            frames.append(h)

robustness_df = pd.concat(frames, ignore_index=True)
summary = summarize_final_rounds(robustness_df, ["dataset", "aggregator", "attack_fraction"])
display(summary)
plot_robustness(robustness_df, ROOT / "results" / "notebook_robustness.png")

## Adaptive DP and CKKS privacy

The publication configuration uses adaptive privacy. DP applies clipping and Gaussian noise and uses the Opacus RDP accountant when available. HE uses TenSEAL CKKS with polynomial modulus degree 8192 and chunked encrypted weighted aggregation.

In [ ]:
adaptive_privacy = PrivacyConfig(
    mode="adaptive",
    clip_norm=1.0,
    noise_multiplier=1.1,
    delta=1e-5,
    sensitivity=0.60,
    he_poly_modulus_degree=8192,
    he_scale_bits=40,
    he_chunk_size=2048,
)

privacy_exp = replace(quick_exp, rounds=5, attack_fraction=0.0, attack="none", seed=45)
privacy_history, _, _ = run_experiment(
    privacy_exp,
    adaptive_privacy,
    AggregationConfig(method="braga"),
    quick_network,
    cache_dir=str(ROOT / "data" / "cache"),
    out_dir=ROOT / "results" / "notebook_adaptive_privacy",
)
display(privacy_history[["round", "privacy_mode", "epsilon", "auc", "f1", "consensus_mode", "consensus_latency_ms"]])

## Measured consensus benchmark

The following cell invokes the signed PBFT, PoA, and hybrid protocol emulator. Throughput is computed from actual successful commits divided by wall clock duration; latency is measured from each protocol execution.

In [ ]:
import subprocess

subprocess.run([
    sys.executable,
    str(ROOT / "scripts" / "benchmark_consensus.py"),
    "--validators", "4", "7", "10", "16",
    "--transactions", "100",
    "--latency-ms", "2",
    "--jitter-ms", "0.5",
    "--packet-loss", "0.0",
    "--output", str(ROOT / "results" / "consensus_benchmark.csv"),
], check=True, env={**__import__("os").environ, "PYTHONPATH": str(ROOT / "src")})

display(pd.read_csv(ROOT / "results" / "consensus_benchmark.csv"))

## Full publication suite

The complete configuration evaluates SECOM, C-MAPSS, Tennessee Eastman, and FactoryNet; FedAvg and BRAGA; malicious client ratios 0, 10, 20, and 30 percent; five seeds; 50 communication rounds; adaptive privacy; and hybrid consensus.

In [ ]:
RUN_FULL_PUBLICATION_SUITE = False

if RUN_FULL_PUBLICATION_SUITE:
    subprocess.run([
        sys.executable,
        str(ROOT / "scripts" / "run_experiments.py"),
        "--config", str(ROOT / "configs" / "publication.yaml"),
        "--output", str(ROOT / "results" / "publication"),
    ], check=True, env={**__import__("os").environ, "PYTHONPATH": str(ROOT / "src")})
    display(pd.read_csv(ROOT / "results" / "publication" / "summary_ci95.csv"))
else:
    print("Set RUN_FULL_PUBLICATION_SUITE=True to execute the complete publication experiment matrix.")

## Module level ablation

The ablation script tests the full design, no blockchain, no privacy, FedAvg instead of BRAGA, PBFT only, and PoA only.

In [ ]:
RUN_ABLATION = False

if RUN_ABLATION:
    subprocess.run([
        sys.executable,
        str(ROOT / "scripts" / "run_ablation.py"),
    ], check=True, cwd=ROOT, env={**__import__("os").environ, "PYTHONPATH": str(ROOT / "src")})
else:
    print("Set RUN_ABLATION=True to run the module level ablation suite.")